# Inference with a fine-tuned Surya checkpoint

This notebook reloads a checkpoint saved by **_2_finetune_template_1D.ipynb_** (or by the
training script) and runs the same validation test that notebook did at its end: forward a
batch, sigmoid + threshold at 0.5, and plot input / ground truth / prediction side by side.

**How loading works.** A Lightning checkpoint stores *weights only* (a `state_dict` keyed by
module names), not the Python objects. So we must rebuild everything exactly as it was built
during training — same config, same `HelioSpectformer2D.from_config(...)`, and crucially the
**same LoRA wrap**, because PEFT renames parameters (`backbone.X` → `base_model.model.backbone.X`)
and the checkpoint keys follow the *wrapped* names. `CHLightningModule` does not save its
`__init__` arguments, so we pass `model=` and `metrics=` to `load_from_checkpoint()` ourselves.

**Run this on the server that has** the checkpoint file and the mask base path
(`ch_mask_base_path` in the config), from the `downstream_apps/test/` directory, with the same
conda environment used for training.

## Set your cuda visible device

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys

import torch
import lightning as L

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the workshop_infrastructure folder.
sys.path.append("../../")

from workshop_infrastructure.utils import build_scalers, apply_peft_lora

torch.set_float32_matmul_precision('medium')

## Load configuration

The same config file used for training. It is the single source of truth for the architecture
(so the rebuilt model matches the checkpoint), the channels, and the dataset paths.

In [ ]:
from downstream_apps.test.configs import load_ch_config

cfg = load_ch_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")

## Download assets

Inference needs **only the scalers**: every weight (pretrained backbone + LoRA adapters + head)
lives in the fine-tuned checkpoint, so the ~1.8 GB backbone download is skipped here.

In [ ]:
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")

## Build the inference dataloader

Same pipeline as in the training notebook (`build_helio_dataloaders` + `CHDSDataset`), so the
inputs are preprocessed/normalized exactly like they were during training — that part of the
code must not diverge, or the model sees data it was never trained on.

To infer on **new samples**, either:
- point `cfg.data.valid_data_path` at a new index CSV *before* this cell (use an absolute
  path — the relative-path resolution against the config dir only happens inside
  `load_config`), or
- raise / remove `max_number_of_samples` below to go beyond the 49 samples used while
  training.

Note the builder returns both loaders; we only use the validation one, which is not shuffled
and runs with `phase="val"` (no random channel masking or flips).

**Caveat:** `CHDSDataset` always loads a ground-truth mask per sample, so genuinely new data
needs mask files (or a small adaptation of `_load_mask`) to flow through this pipeline.

In [ ]:
from workshop_infrastructure.datasets.builders import build_helio_dataloaders
from downstream_apps.test.datasets.ch_dataset import CHDSDataset

# cfg.data.valid_data_path = "/absolute/path/to/my_new_index.csv"  # <- uncomment & edit for new samples

# Same cap as in training so results are comparable; raise it (or set None) for more samples.
max_number_of_samples = 49
batch_size = cfg.batch_size

_, inference_data_loader = build_helio_dataloaders(
    cfg,
    CHDSDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=max_number_of_samples,
    ds_ch_index_path=cfg.data.ch_index_path,
    ds_ch_mask_base_path=cfg.data.ch_mask_base_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

print(f"inference set: {len(inference_data_loader.dataset)} samples | batch_size: {batch_size}")

## Rebuild the model exactly as it was trained

Identical to the corresponding cells of the training notebook: `from_config()` maps the
`model:` section of the config onto the architecture arguments, then the same freeze / LoRA
logic is applied.

The LoRA wrap in particular **must** happen before loading the checkpoint — PEFT renames the
parameters, so the checkpoint's keys only line up with the wrapped model. Skipping it (or
changing `lora_config`) makes the load fail with missing/unexpected keys (`strict=True`).

No `load_pretrained_weights()` here: the fine-tuned checkpoint already contains every weight,
so re-downloading and re-loading the pretrained backbone would be wasted work.

In [ ]:
from workshop_infrastructure.models.finetune_models import HelioSpectformer2D

model = HelioSpectformer2D.from_config(
    cfg.model,
    dtype=cfg.dtype,
    ft_out_chans=1,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
)

# Same freeze / LoRA logic as during training (selected from the model: section of the config).
if cfg.model.freeze_backbone:
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            param.requires_grad = False

if cfg.model.use_lora:
    model = apply_peft_lora(model, cfg.model.lora_config)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Load the checkpoint into a Lightning module

`CHLightningModule` does not save its `__init__` arguments into the checkpoint (there is no
`save_hyperparameters()` in it), so we hand `load_from_checkpoint()` the freshly rebuilt model
and the metrics dict; Lightning then restores every weight from the checkpoint's `state_dict`.

In [ ]:
from downstream_apps.test.metrics.template_metrics import CHThresholdMetrics
from downstream_apps.test.lightning_modules.pl_simple_baseline import CHLightningModule

metrics = {
    'train_loss': CHThresholdMetrics("train_loss"),
    'val_loss': CHThresholdMetrics("val_loss"),
    'train_metrics': CHThresholdMetrics("train_metrics"),
    'val_metrics': CHThresholdMetrics("val_metrics"),
}

In [ ]:
# Path (on this server) to the checkpoint written by the training run.
ckpt_path = "./wandb/wandb_tmp/coronal_hole_segmentation/dgeacv0d/checkpoints/epoch=11-step=240.ckpt"

lit_model = CHLightningModule.load_from_checkpoint(
    ckpt_path,
    model=model,          # the freshly rebuilt (and LoRA-wrapped) model
    metrics=metrics,
    lr=cfg.learning_rate,
    batch_size=batch_size,
    map_location="cpu",   # load to CPU first, then move where we want
    strict=True,          # fail loudly if the architecture does not match the checkpoint
)

# eval() disables dropout etc. — the backbone was built with drop_rate > 0, so this matters.
lit_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
lit_model.to(device)
print(f"Checkpoint loaded onto {device}.")

In [ ]:
# Optional sanity check: every tensor of the rebuilt module should have been restored from
# the checkpoint — no missing keys, no unexpected keys, no shape mismatches. A non-empty
# list here means the reconstruction above does not match the model that was trained
# (e.g. LoRA not applied, or a different config).
ckpt_state = torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"]
model_state = lit_model.state_dict()

missing = sorted(set(model_state) - set(ckpt_state))
unexpected = sorted(set(ckpt_state) - set(model_state))
shape_mismatches = sorted(
    k for k in model_state if k in ckpt_state and model_state[k].shape != ckpt_state[k].shape
)

print(f"restored: {len(model_state) - len(missing)} / {len(model_state)} tensors")
print(f"missing: {missing[:5]}{' ...' if len(missing) > 5 else ''}")
print(f"unexpected: {unexpected[:5]}{' ...' if len(unexpected) > 5 else ''}")
print(f"shape mismatches: {shape_mismatches[:5]}")

## Run inference on a batch

The same test as at the end of the training notebook: grab one (unshuffled) batch, move it to
the model's device, forward it, and turn the logits into a binary mask with a 0.5 sigmoid
threshold. `CHLightningModule.forward` already squeezes the singleton output-channel dim, so
`logits` is `(B, H, W)`, matching the `(B, H, W)` masks.

In [ ]:
# Grab one inference batch (already preprocessed/normalized exactly like training)
inference_batch = next(iter(inference_data_loader))

# Move the batch's tensors to whatever device the model is on
inference_batch_on_device = {
    k: (v.to(device) if torch.is_tensor(v) else v)
    for k, v in inference_batch.items()
}

In [ ]:
with torch.no_grad():
    logits = lit_model(inference_batch_on_device)   # (B, H, W)
    probs = torch.sigmoid(logits)
    pred_mask = (probs > 0.5).float()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pick which sample in the batch to inspect
sample_idx = 0

# Locate the aia193 channel using the config, instead of hardcoding its index
aia193_idx = cfg.data.channels.index("aia193")

# batch["ts"] shape: (B, C, T, H, W) — we only have one input timestep (T=1) here
aia193_img = inference_batch["ts"][sample_idx, aia193_idx, ...].squeeze().detach().cpu().numpy()
predicted_mask = pred_mask[sample_idx, ...].detach().cpu().numpy()
# Ground-truth mask for the same sample, same shape convention as pred_mask: (H, W)
ground_truth_mask = inference_batch["mask"][sample_idx, ...].squeeze().detach().cpu().numpy()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin = np.percentile(aia193_img, 1)
vmax = np.percentile(aia193_img, 99.8)

axes[0].imshow(aia193_img, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("AIA 193 (input)")
axes[0].axis("off")

axes[0].contour(
    ground_truth_mask,
    levels=[0.5],
    colors="lime",
    linewidths=1.0,
    origin="lower",
)

axes[1].imshow(ground_truth_mask, origin="lower", cmap="gray")
axes[1].set_title("Ground-truth CH mask")
axes[1].axis("off")

axes[2].imshow(predicted_mask, origin="lower", cmap="gray")
axes[2].set_title("Predicted CH mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Score the predictions

Same metrics object that logged `val_metrics` during training (hard IoU and Dice on the 0.5-thresholded mask), applied here to the batch above, then averaged over the whole inference set.

In [ ]:
val_metrics = CHThresholdMetrics("val_metrics")

metric_dict, _ = val_metrics(logits, inference_batch_on_device["mask"].float())
print({k: float(v) for k, v in metric_dict.items()})

In [ ]:
ious, dices = [], []
with torch.no_grad():
    for batch in inference_data_loader:
        batch_on_device = {
            k: (v.to(device) if torch.is_tensor(v) else v)
            for k, v in batch.items()
        }
        logits = lit_model(batch_on_device)
        metric_dict, _ = val_metrics(logits, batch_on_device["mask"].float())
        ious.append(float(metric_dict["iou"]))
        dices.append(float(metric_dict["dice_coef"]))

print(f"Mean IoU  over {len(ious)} batches: {np.mean(ious):.4f}")
print(f"Mean Dice over {len(dices)} batches: {np.mean(dices):.4f}")

## Conclusion

The checkpointed model is fully restored (backbone + LoRA adapters + head) and can now be used
for inference on any batch the `CHDSDataset` pipeline can produce. Remember:
- The rebuild before `load_from_checkpoint()` must stay in lockstep with the training config
  (especially the LoRA wrap).
- New samples = a new index CSV (`cfg.data.valid_data_path`) and/or a larger
  `max_number_of_samples`; masks are still required by the current dataset class.